# NovaMarket — Kod Bilenler İçin: Basit Tahmin Modeli

**Hazırlayan:** Fatmanur Erdem
**Veri seti:** NovaMarket_Clean.csv (Haziran 2026)

Bu notebook, ortak keşif bölümünde (1-3. Gün) tasarlanan tahmin problemini basit bir Python modeliyle deniyor. Amaç gelişmiş bir model kurmak değil, kağıt üzerinde yapılan düşünme sürecinin kodda nasıl göründüğünü deneyimlemek.

## 1. Problem

**Orijinal problem (ortak keşif, Gün 3):** "Önümüzdeki ay hangi ürün kategorisinin (Ofis, Elektronik, Spor, Kitap, Kozmetik, Ev) cirosu bir önceki aya göre artar mı (Artar/Artmaz)?"

**İş değeri:** Bu tahmin, Satın Alma / Stok Yönetimi ekibinin hangi kategoriye daha fazla stok, bütçe ve raf/kampanya alanı ayıracağı kararına yardımcı olabilir.

**Neden bire bir aynı problemi kodlayamıyorum:** Veri seti yalnızca Haziran 2026'yı (tek bir ay) kapsıyor. Aylar arası bir karşılaştırma yapabilmek için en az 2 ayın gerçek verisine ihtiyaç var, ama elimde sadece 1 ay var — bu, Gün 3'ün kapanış notunda zaten öngörülen bir sınırlamaydı ("yeterli geçmiş veri olmaması").

**Kodlama için kullandığım proxy (vekil) problem:** Aynı sorunun mantığını, veri setinin *kendi içinde* test edilebilir hale getirmek için ay yerine **hafta** birimini kullanıyorum: *"Bir ürün kategorisinin bu haftaki cirosu, bir önceki haftaya göre artar mı (Artar/Artmaz)?"* Haziran ayı 4 tam haftaya bölünüyor (5. hafta sadece 2 gün olduğu için hariç tutuldu), bu da 6 kategori × 3 hafta geçişi = **18 gözlem** veriyor. Örneklem yine çok küçük — bunu dürüstçe not ediyorum ve sonuçları buna göre yorumluyorum.

## 2. Veriyi Aç

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('NovaMarket_Clean.csv')
print('Satır sayısı:', len(df))
print('Kolon sayısı:', df.shape[1])
df.head()

Satır sayısı: 3500
Kolon sayısı: 19


,Order_ID,Order_Date,Day_Name,Order_Time,City,Store,Customer_ID,Customer_Type,Gender,Age,Product_Category,Product_Name,Quantity,Unit_Price,Discount,Payment_Method,Employee,Channel,Revenue
0,ORD500001,2026-06-11,Thursday,21:30,Antalya,Muratpaşa,CUST10869,New,Female,20.0,Ofis,Defter,1,179,0,PayPal,EMP_08,Web,179.0
1,ORD500002,2026-06-21,Sunday,18:00,Bursa,Nilüfer,CUST10211,New,Male,52.0,Elektronik,Powerbank,2,899,15,Wallet,EMP_04,Mobile,1528.3
2,ORD500003,2026-06-18,Thursday,19:45,İstanbul,Kadıköy,CUST10499,New,Male,47.0,Spor,Koşu Çantası,1,499,0,Bank Transfer,EMP_17,Mobile,499.0
3,ORD500004,2026-06-16,Tuesday,20:55,İzmir,Bornova,CUST11792,Returning,Male,44.0,Elektronik,Kablosuz Mouse,1,499,0,Card,EMP_23,Mobile,499.0
4,ORD500005,2026-06-12,Friday,18:45,Konya,Selçuklu,CUST11369,Returning,Female,62.0,Ofis,Masa Düzenleyici,1,399,10,PayPal,EMP_15,Web,359.1


## 3. Target ve Feature

**Target:** `target_increase` — kategorinin bu haftaki cirosu bir önceki haftaya göre arttı mı? (1 = Artar, 0 = Artmaz)

**Feature'lar (bir önceki haftadan, yani tahmin anında zaten bilinen bilgiler):**
- `prev_revenue` — önceki haftanın toplam cirosu
- `prev_quantity` — önceki haftanın toplam satış adedi
- `prev_discount_rate` — önceki hafta indirimli sipariş oranı
- `prev_mobile_share` — önceki hafta Mobile kanal payı
- `prev_returning_share` — önceki hafta tekrar eden müşteri oranı

Not: `Employee`, `Order_ID`, `Customer_ID` gibi ortak keşif bölümünde (Gün 3, Görev 6) "Hayır" işaretlediğim bilgileri buraya da almadım — kategori düzeyinde anlamlı bir sinyal taşımıyorlar. `Product_Category`'yi de bilerek feature olarak eklemedim: amaç modelin belirli bir kategoriyi ezberlemesi değil, hafta içindeki genel momentum örüntüsünü öğrenmesi.

In [2]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['week'] = ((df['Order_Date'].dt.day - 1) // 7) + 1
df = df[df['week'] <= 4]  # 5. hafta sadece 2 gün (29-30 Haziran), tam olmadığı için hariç

# Kategori x hafta bazında özet
agg = df.groupby(['Product_Category', 'week']).agg(
    revenue=('Revenue', 'sum'),
    quantity=('Quantity', 'sum'),
    discount_rate=('Discount', lambda s: (s > 0).mean()),
    mobile_share=('Channel', lambda s: (s == 'Mobile').mean()),
    returning_share=('Customer_Type', lambda s: (s == 'Returning').mean()),
).reset_index()

# Hafta geçişlerini satır haline getir: önceki hafta -> bu hafta
rows = []
for cat, g in agg.groupby('Product_Category'):
    g = g.sort_values('week').reset_index(drop=True)
    for i in range(1, len(g)):
        prev, curr = g.iloc[i - 1], g.iloc[i]
        rows.append({
            'category': cat, 'week_from': int(prev['week']), 'week_to': int(curr['week']),
            'prev_revenue': prev['revenue'], 'prev_quantity': prev['quantity'],
            'prev_discount_rate': prev['discount_rate'], 'prev_mobile_share': prev['mobile_share'],
            'prev_returning_share': prev['returning_share'],
            'curr_revenue': curr['revenue'],
            'target_increase': int(curr['revenue'] > prev['revenue']),
        })
data = pd.DataFrame(rows)
print('Gözlem (satır) sayısı:', len(data))
print('Target dağılımı:', data['target_increase'].value_counts().to_dict())
data

Gözlem (satır) sayısı: 18
Target dağılımı: {0: 11, 1: 7}


,category,week_from,week_to,prev_revenue,prev_quantity,prev_discount_rate,prev_mobile_share,prev_returning_share,curr_revenue,target_increase
0,Elektronik,1,2,541358.65,402,0.483721,0.572093,0.479070,486623.50,0
1,Elektronik,2,3,486623.50,394,0.556604,0.632075,0.452830,490432.70,1
2,Elektronik,3,4,490432.70,411,0.520548,0.589041,0.497717,590278.25,1
3,Ev,1,2,225024.95,273,0.541985,0.580153,0.419847,248862.30,1
4,Ev,2,3,248862.30,276,0.582090,0.574627,0.514925,248820.85,0
5,Ev,3,4,248820.85,301,0.496732,0.601307,0.398693,246408.40,0
6,Kitap,1,2,69333.90,175,0.434783,0.673913,0.478261,65338.25,0
7,Kitap,2,3,65338.25,166,0.518072,0.602410,0.373494,74235.80,1
8,Kitap,3,4,74235.80,192,0.566038,0.622642,0.518868,66326.25,0
9,Kozmetik,1,2,159523.40,229,0.452381,0.658730,0.500000,194003.95,1


## 4. Eğitim ve Test

Veriyi eğitim (train) ve test setlerine ayırıyorum. 18 gözlem çok küçük bir veri seti olduğu için 6 gözlemi test için ayırıyorum (yaklaşık %33), `stratify` ile her iki sınıfın da test setinde temsil edilmesini sağlıyorum.

In [3]:
from sklearn.model_selection import train_test_split

feature_cols = ['prev_revenue', 'prev_quantity', 'prev_discount_rate', 'prev_mobile_share', 'prev_returning_share']
X = data[feature_cols]
y = data['target_increase']

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, data.index, test_size=6, stratify=y, random_state=42
)
print('Eğitim seti:', len(X_train), 'satır | Test seti:', len(X_test), 'satır')
print('Eğitimdeki target dağılımı:', y_train.value_counts().to_dict())
print('Testteki target dağılımı:', y_test.value_counts().to_dict())

Eğitim seti: 12 satır | Test seti: 6 satır
Eğitimdeki target dağılımı: {0: 7, 1: 5}
Testteki target dağılımı: {0: 4, 1: 2}


## 5. Baseline

En basit yöntem: eğitim setinde en sık görülen sınıfı, test setindeki her gözlem için tahmin olarak vermek.

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy='most_frequent', random_state=42)
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)

print('Baseline tahminleri:', list(baseline_pred))
print('Baseline doğruluk (accuracy):', round(baseline_acc, 3))

Baseline tahminleri: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
Baseline doğruluk (accuracy): 0.667


## 6. Model

Target ikili (Artar/Artmaz) olduğu için bu bir **classification** problemi. Rehberdeki öneriye uyarak **Logistic Regression** kullanıyorum. Feature'ların ölçekleri çok farklı olduğu için (ciro binlerce TL, oranlar 0-1 arası) önce `StandardScaler` ile ölçeklendiriyorum.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)
model_pred = model.predict(X_test_s)
model_acc = accuracy_score(y_test, model_pred)

print('Model tahminleri:', list(model_pred))
print('Model doğruluk (accuracy):', round(model_acc, 3))

Model tahminleri: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
Model doğruluk (accuracy): 0.667


## 7. Karşılaştırma

In [6]:
print(f"Baseline accuracy : {baseline_acc:.3f}")
print(f"Model accuracy    : {model_acc:.3f}")
print()
if model_acc > baseline_acc:
    print("Model, baseline'dan daha iyi.")
elif model_acc == baseline_acc:
    print("Model, baseline ile AYNI sonucu verdi — bu veri boyutuyla modelin baseline'dan bir üstünlüğü kanıtlanamadı.")
else:
    print("Model, baseline'dan daha kötü.")

Baseline accuracy : 0.667
Model accuracy    : 0.667

Model, baseline ile AYNI sonucu verdi — bu veri boyutuyla modelin baseline'dan bir üstünlüğü kanıtlanamadı.


## 8. Örnek Tahminler (Test Setindeki Tüm 6 Gözlem)

In [7]:
result = data.loc[idx_test, ['category', 'week_from', 'week_to', 'target_increase']].copy()
result = result.rename(columns={'target_increase': 'gercek_sonuc'})
result['baseline_tahmin'] = baseline_pred
result['model_tahmin'] = model_pred
result['model_olasiligi'] = model.predict_proba(X_test_s)[:, 1].round(2)
result['model_dogru_mu'] = (result['gercek_sonuc'] == result['model_tahmin'])
result

,category,week_from,week_to,gercek_sonuc,baseline_tahmin,model_tahmin,model_olasiligi,model_dogru_mu
8,Kitap,3,4,0,0,0,0.12,True
11,Kozmetik,3,4,0,0,0,0.02,True
12,Ofis,1,2,0,0,0,0.15,True
7,Kitap,2,3,1,0,0,0.37,False
9,Kozmetik,1,2,1,0,0,0.02,False
6,Kitap,1,2,0,0,0,0.01,True


## 9. Yorum

**Doğru bir tahmin örneği:** Kitap kategorisi, 3. haftadan 4. haftaya geçişte gerçek sonuç "Artmaz" (0) idi ve model de doğru şekilde "Artmaz" tahmini yaptı (olasılık ≈ 0.12, yani modelin de artış beklemediği görülüyor). Bu ve benzeri düşüş/durağan haftalarda model, çoğunluk sınıfını doğru yakaladı.

**Yanlış bir tahmin örneği:** Kozmetik kategorisi, 1. haftadan 2. haftaya geçişte gerçek sonuç "Artar" (1) idi (ciro 159k'dan 194k'ya çıkmıştı) ama model "Artmaz" (0) tahmini yaptı — üstelik çok düşük bir artış olasılığı (≈0.02) verdi. Model, bu haftadaki artışı yakalayacak bir sinyali feature'larda bulamadı; muhtemelen çünkü eğitim setinde sadece 12 gözlem vardı ve model çoğunluk sınıfına ("Artmaz") aşırı güvenmeyi öğrendi.

**Genel gözlem:** Model, test setinde baseline ile birebir aynı tahminleri üretti (hepsi "Artmaz"). Bu, modelin gerçek bir örüntü öğrenmediğini, sadece en güvenli/en sık seçeneği tekrarladığını gösteriyor — 12 satırlık bir eğitim setiyle 5 feature'lı bir model için beklenen bir sonuç. Bu durum, Gün 4'te elle yapılan tahmin denemesinde ulaşılan sonuçla da tutarlı: gerçek bir doğrulama için çok daha fazla geçmiş veriye ihtiyaç var.

## Kod Çalışmasının Çıktısı — Sorular

**Hangi target'ı tahmin ettin?**
Bir ürün kategorisinin bu haftaki cirosunun bir önceki haftaya göre artıp artmadığı (Artar=1 / Artmaz=0). Bu, ortak keşif bölümündeki "aylık" hedefin, tek aylık veriyle test edilebilir hale getirilmiş hafta bazlı bir vekilidir.

**Hangi bilgileri kullandın?**
Önceki haftanın toplam cirosu, toplam satış adedi, indirimli sipariş oranı, Mobile kanal payı ve tekrar eden müşteri oranı. Kategori adını kasıtlı olarak feature yapmadım (genelleme yapabilsin diye), Employee/Order_ID/Customer_ID gibi sinyal taşımayan kolonları da ortak keşif bölümündeki gerekçeyle dışarıda bıraktım.

**Baseline sonucu neydi?**
%66,7 doğruluk (6 test gözleminden 4'ü) — her zaman en sık görülen sınıfı ("Artmaz") tahmin ederek.

**Model sonucu neydi?**
%66,7 doğruluk — baseline ile birebir aynı, çünkü model de test setindeki her gözlem için "Artmaz" tahmini yaptı.

**Model en basit yöntemden daha iyi miydi?**
Hayır. Bu veri boyutuyla (18 gözlem, 12 eğitim) modelin baseline'a göre bir üstünlüğü kanıtlanamadı — model gerçek bir örüntü öğrenmek yerine çoğunluk sınıfını tekrarladı.

**En dikkat çekici hata neydi?**
Model, gerçekte ciro artışı olan iki haftayı da ("Kitap" 2→3, "Kozmetik" 1→2) kaçırdı — ikisinde de çok düşük artış olasılığı verdi. Yani modelin hatası sistematik bir yönde: artışları (pozitif sınıfı) neredeyse hiç yakalayamıyor. Bu, Görev 10'daki "yanlış negatif daha maliyetli olabilir" tespitiyle doğrudan örtüşüyor — gerçek bir kullanımda bu model tam da en riskli hata türünü yapıyor.

**Bu denemeden ne öğrendin?**
Kodun çalışması ve bir "accuracy" sayısı üretmesi tek başına bir şey kanıtlamıyor. Burada model teknik olarak doğru kuruldu (train/test ayrımı, ölçekleme, baseline karşılaştırması hepsi yapıldı) ama sonuç dürüstçe söylemek gerekirse "işe yaramadı" — ve bunun nedeni model seçimi değil, örneklem büyüklüğü. Bu, ortak keşif bölümünde defalarca vurgulanan "yeterli geçmiş veri olmadan güvenilir bir tahmin kurulamaz" fikrinin kod tarafında somut biçimde doğrulanması oldu. Gerçek bir sonraki adım, en az birkaç aylık veri toplayıp bu pipeline'ı (feature'lar, baseline, model, karşılaştırma) aynen tekrar çalıştırmak olurdu.